In [1]:
import rich
import langgraph

In [2]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict

# Define your state shape
class State(TypedDict):
    messages: list[str]
    step: int

In [3]:
# Define simple nodes
def process_step_1(state: State) -> State:
    """First step: analyze input"""
    print("  → Step 1: Processing input...")
    import time
    time.sleep(0.5)  # Simulate work
    return {"step": 1, "messages": state["messages"] + ["step_1_done"]}

def process_step_2(state: State) -> State:
    """Second step: validate"""
    print("  → Step 2: Validating...")
    import time
    time.sleep(0.5)
    return {"step": 2, "messages": state["messages"] + ["step_2_done"]}

def process_step_3(state: State) -> State:
    """Third step: execute"""
    print("  → Step 3: Executing...")
    import time
    time.sleep(0.5)
    return {"step": 3, "messages": state["messages"] + ["step_3_done"]}

In [4]:
# Build the graph
builder = StateGraph(State)
builder.add_node("step_1", process_step_1)
builder.add_node("step_2", process_step_2)
builder.add_node("step_3", process_step_3)

# Connect nodes sequentially
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")

# Set entry and exit
builder.set_entry_point("step_1")
builder.set_finish_point("step_3")

# Compile the graph
graph = builder.compile()

In [5]:
from rich.console import Console

console = Console()
initial_input = {"messages": ["Mark my task as done"], "step": 0}

# 1. Start the UI Spinner
with console.status("[bold blue]Initializing mini graph...", spinner="dots") as status:
    
    # 2. Open the Stream
    for event in graph.stream(initial_input):
        
        # 3. Extract the Node Name from the yielded event
        for node_name, state_update in event.items():
            
            # 4. Update the UI based on which node just fired
            if node_name == "step_1":
                status.update("[bold yellow]🔍 Analyzing input...")
                
            elif node_name == "step_2":
                status.update("[bold cyan]✓ Validating data...")
                
            elif node_name == "step_3":
                status.update("[bold green]⚡ Executing task...")

# 5. The stream is over. The spinner automatically disappears.
console.print(f"\n[bold green]✅ Complete![/bold green]")
console.print(f"Final state: {state_update}")
console.print(f"Messages: {state_update['messages']}")

→ Step 1: Processing input...

Output()

→ Step 2: Validating...

→ Step 3: Executing...

✅ Complete!

Final state: {'step': 3, 'messages': ['Mark my task as done', 'step_1_done', 'step_2_done', 'step_3_done']}

Messages: ['Mark my task as done', 'step_1_done', 'step_2_done', 'step_3_done']

In [6]:
# Explore rich console features
console.print("\n[bold magenta]═══ Rich Console Features ═══[/bold magenta]")
console.print("✨ You can use emojis, colors, and styling")
console.print("[red]Error message[/red]")
console.print("[yellow]Warning message[/yellow]")
console.print("[green]Success message[/green]")

# Try a table
from rich.table import Table

table = Table(title="Mini Graph Events")
table.add_column("Node", style="cyan")
table.add_column("Status", style="magenta")

for msg in state_update["messages"]:
    table.add_row(msg, "✓ Done")

console.print(table)

═══ Rich Console Features ═══

✨ You can use emojis, colors, and styling

Error message

Warning message

Success message

        Mini Graph Events        
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Node                 ┃ Status ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Mark my task as done │ ✓ Done │
│ step_1_done          │ ✓ Done │
│ step_2_done          │ ✓ Done │
│ step_3_done          │ ✓ Done │
└──────────────────────┴────────┘

In [7]:
# Try a progress bar
from rich.progress import track
import time

console.print("\n[bold]Testing progress bar:[/bold]")
for i in track(range(5), description="Processing..."):
    time.sleep(0.3)

Testing progress bar:

Output()

In [14]:
from rich.pretty import pprint, install
from rich import print as rprint
install()

pprint(state_update)
rprint(state_update)

{'step': 3, 'messages': ['Mark my task as done', 'step_1_done', 'step_2_done', 'step_3_done']}

{'step': 3, 'messages': ['Mark my task as done', 'step_1_done', 'step_2_done', 'step_3_done']}

In [13]:
from rich import traceback
traceback.install()

raise RuntimeError("This is a test error to demonstrate rich error formatting")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 from rich import traceback                                                                   │
│   2 traceback.install()                                                                          │
│   3                                                                                              │
│ ❱ 4 raise RuntimeError("This is a test error to demonstrate rich error formatting")              │
│   5                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
RuntimeError: This is a test error to demonstrate rich error formatting